# A2 - BDD100K: Faster R-CNN với C5 và FPN

**CO5085 · Học kỳ 261 · Notebook độc lập.** Mọi định nghĩa, kiểm tra,
train/evaluate và xuất kết quả nằm trong file này. Chạy từ kernel Python 3.14.
BDD100K local có 10 lớp detection; không yêu cầu chạy notebook khác.

**Restart Kernel → Run All** chạy toàn bộ dữ liệu, train mới trên CUDA,
chọn checkpoint bằng validation và đánh giá test. Checkpoint giữ trong RAM;
bảng/đồ thị hiển thị trong output cell. Save notebook để giữ kết quả.

**Đóng góp học thuật:** đọc, kiểm chứng và điều chỉnh code; bổ sung phân tích từ
kết quả thật. Ghi rõ AI đã hỗ trợ phần nào trước khi nộp (mục 1.6 của đề).

Đọc `datasets/bdd100k/{train,val,test}` đã giải nén và kiểm tra. Đây là nhãn
legacy người dùng cung cấp, có test JSON; không nhận là benchmark Detection 2020.
Giữ nguyên JPEG/JSON. Notebook tự tạo cache detection nhỏ từ JSON để tránh
parse lại polygon mỗi epoch; không tải lại archive đã xóa.


## 1. Môi trường và cấu hình

In [ ]:
import copy
import csv
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import random
import time
from concurrent.futures import ThreadPoolExecutor
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
from IPython.display import Markdown, display
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

torch.set_num_threads(min(8, os.cpu_count() or 1))
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
    }
)

In [ ]:
NAME = "A2"
MODE = "full"  # Run All uses the full dataset.  # smoke / pilot / full
SEED = 42
EARLY_STOP_PATIENCE = 4
EARLY_STOP_MIN_DELTA = 1e-4
if not torch.cuda.is_available():
    raise RuntimeError("Chọn kernel Python 3.14 có PyTorch CUDA; notebook yêu cầu GPU.")
DEVICE = torch.device("cuda:0")

NUM_WORKERS = 0  # Windows notebooks: no spawned workers with local Dataset classes
# Pilot evidence must never overwrite final report tables
RUN_TEST = True  # Run once after the validation protocol is fixed
ROOT = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "assignment" / "exercise-vne.pdf").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the CO5085 project folder.")
DATASET_ID = "bdd100k_legacy"
RUN_KEY = Path(NAME)  # Logical RAM namespace; never a directory on disk.
RECORDS = {}
CHECKPOINTS = {}


assert MODE in {"smoke", "pilot", "full"}
VERSIONS = {
    p: importlib.metadata.version(p)
    for p in ["torch", "torchvision", "numpy", "matplotlib", "Pillow", "pycocotools"]
}
print(NAME, MODE, DEVICE, VERSIONS)
MICRO_BATCH = 2
ACCUM = 2
RUN_OVERFIT_CHECK = MODE == "smoke"  # Technical evidence, never report metric
print(
    f"EARLY STOPPING: validation AP50:95; patience={EARLY_STOP_PATIENCE}, min_delta={EARLY_STOP_MIN_DELTA:g}",
    flush=True,
)

## 2. Tiện ích nội bộ

In [ ]:
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False


def digest(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, default=str).encode()
    ).hexdigest()


def write_cache_metadata(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + f".{os.getpid()}.tmp")
    temp.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
    )
    temp.replace(path)


seed_all(SEED)


def snapshot(value):
    """Freeze CPU tensor/array copies so later training cannot change a best state."""
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, np.ndarray):
        return value.copy()
    if isinstance(value, dict):
        return {key: snapshot(item) for key, item in value.items()}
    if isinstance(value, list):
        return [snapshot(item) for item in value]
    if isinstance(value, tuple):
        return tuple(snapshot(item) for item in value)
    return copy.deepcopy(value)


def record_result(key, value):
    """Store protocol, histories and metrics in this kernel; key is not a disk path."""
    RECORDS[Path(key).as_posix()] = snapshot(value)


def get_record(key):
    return RECORDS[Path(key).as_posix()]


def remember_checkpoint(key, state):
    """Keep only best weights in CPU RAM, independently of model storage."""
    CHECKPOINTS[Path(key).as_posix()] = snapshot(state)


def load_checkpoint(key):
    """Read this Run All's best state; no filesystem or other notebook is consulted."""
    return CHECKPOINTS[Path(key).as_posix()]


def show_figure(fig, stem):
    if fig.get_layout_engine() is None:
        fig.tight_layout()
    print(f"[HÌNH] {stem} · hiển thị trong output cell", flush=True)
    plt.show()
    plt.close(fig)


def show_table(rows, columns, title):
    """Render tables as saved cell outputs without creating CSV/LaTeX files."""

    def format_value(value):
        if isinstance(value, (float, np.floating)):
            return f"{value:.4f}"
        return str(value).replace("|", r"\|").replace("\n", " ")

    lines = [
        f"**{title}** (accuracy/F1/AP dưới dạng 0–1 khi tên cột không ghi %)",
        "",
        "| " + " | ".join(columns) + " |",
        "|" + "|".join("---" for _ in columns) + "|",
    ]
    for row in rows:
        lines.append(
            "| " + " | ".join(format_value(row.get(c, "—")) for c in columns) + " |"
        )
    display(Markdown("\n".join(lines)))


def finish_notebook():
    CHECKPOINTS.clear()
    print(
        "[HOÀN TẤT] Kết quả/bảng/biểu đồ nằm trong output cell. "
        "Lưu file .ipynb để giữ kết quả; không tạo file output hay tự xuất assets LaTeX.",
        flush=True,
    )

In [ ]:
from matplotlib.patches import Rectangle
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torchvision.ops import (
    batched_nms,
    box_area,
    box_iou,
    clip_boxes_to_image,
    roi_align,
)
from torchvision.ops.misc import FrozenBatchNorm2d

## 3. Dataset, nguồn, quyền sử dụng và protocol
[BDD100K, CVPR2020](https://openaccess.thecvf.com/content_CVPR_2020/html/Yu_BDD100K_A_Diverse_Driving_Dataset_for_Heterogeneous_Multitask_Learning_CVPR_2020_paper.html),
[định dạng](https://github.com/bdd100k/bdd100k/blob/master/doc/source/format.rst),
[điều kiện sử dụng](https://github.com/bdd100k/bdd100k/blob/master/doc/source/license.rst).
Giữ dữ liệu local và điều kiện nguồn; giấy phép toolkit không thay giấy phép dataset.

Hai archive người dùng đã được kiểm tra CRC, SHA256, JPEG/JSON và đồng bộ tên.
Schema legacy `name/frames/objects` gồm bbox và polygon: chỉ `box2d` dùng detection.
Ánh xạ `person→pedestrian`, `bike→bicycle`, `motor→motorcycle`, còn lại giữ tên.
10 lớp gồm pedestrian, rider, car, truck, bus, train, motorcycle, bicycle,
traffic light, traffic sign; head có 11 logits tính cả background.

Raw split 70k/10k/20k. Loại trùng SHA256 theo ưu tiên test > val > train; trong
split giữ tên nhỏ nhất: train **69.998**, dev **9.998**, test **20.000**. Một ảnh
train trùng test bị loại khỏi train. Raw file giữ nguyên; danh sách lọc và hash
giữ trong RECORDS (RAM). Run All dùng toàn bộ split đã lọc. Cùng subset giữa bốn run. Test chỉ evaluate sau khi khóa protocol.

Theo quy ước Scalabel inclusive, chuyển `[x1,y1,x2,y2]` thành liên tục
`[x1,y1,x2+1,y2+1]`, clamp trong kích thước ảnh. Box source có endpoint trùng được giữ nếu còn diện tích trong ảnh sau chuyển đổi.
Box nằm đúng ngoài mép ảnh (x1=x2=width hoặc y1=y2=height) không còn diện tích
sau clamp: loại riêng và ghi đầy đủ source box/lý do trong detection_cache/*.json.
Không tạo bbox giả bằng cách kéo tọa độ vào ảnh. Occluded/truncated
vẫn là positive; chỉ crowd/ignored tường minh mới dùng iscrowd.
Metric chính COCO-style AP50:95, kèm AP50/AP75/size/AR100, per-class AP và PR.
Nhãn test local chưa được xác nhận tương thích Detection 2020; kết quả local
không được gọi là official benchmark score.


In [ ]:
CLASS_NAMES = [
    "background",
    "pedestrian",
    "rider",
    "car",
    "truck",
    "bus",
    "train",
    "motorcycle",
    "bicycle",
    "traffic light",
    "traffic sign",
]
NUM_CLASSES = len(CLASS_NAMES)
CATEGORY = {name: i for i, name in enumerate(CLASS_NAMES) if i}
ALIASES = {"person": "pedestrian", "bike": "bicycle", "motor": "motorcycle"}
CACHE = Path(os.environ.get("BDD_ROOT", str(ROOT / "datasets" / "bdd100k")))
verification_path = CACHE / "verification.json"
if not verification_path.is_file():
    raise FileNotFoundError(
        f"Prepared BDD100K missing: {verification_path}. BDD_ROOT must contain train/val/test and verification.json."
    )
VERIFICATION = json.loads(verification_path.read_text(encoding="utf-8"))
assert VERIFICATION["schema"] == 1 and VERIFICATION["error_count"] == 0
ANNOTATION_PROTOCOL = {
    "schema": 2,
    "dataset": DATASET_ID,
    "classes": CLASS_NAMES,
    "aliases": ALIASES,
    "coordinates": "inclusive_xyxy_to_exclusive_plus1_clamped_drop_no_visible_area",
    "ignore": "explicit_crowd_or_ignored_only; occluded/truncated_retained",
}


def parse_bdd_annotation(annotation, width, height, expected_name=None):
    """Legacy one-frame annotation; source JSON is never rewritten."""
    if expected_name is not None and Path(annotation["name"]).stem != expected_name:
        raise ValueError(f"Annotation name mismatch: {expected_name}")
    frames = annotation["frames"]
    if len(frames) != 1:
        raise ValueError("Detection input must have exactly one annotated frame")
    boxes, labels, crowd, dropped = [], [], [], []
    for obj in frames[0]["objects"]:
        if "box2d" not in obj:
            continue  # Polygon-only lane/drivable annotations are not detections.
        category = ALIASES.get(obj["category"], obj["category"])
        if category not in CATEGORY:
            raise ValueError(f"Unknown bbox category: {obj['category']}")
        bbox = obj["box2d"]
        x1, y1, x2, y2 = [float(bbox[k]) for k in ["x1", "y1", "x2", "y2"]]
        if not all(math.isfinite(x) for x in [x1, y1, x2, y2]) or x2 < x1 or y2 < y1:
            raise ValueError(f"Invalid source box: {bbox}")
        box = [max(0, x1), max(0, y1), min(width, x2 + 1), min(height, y2 + 1)]
        box = np.asarray(box, dtype=np.float32).tolist()
        if box[2] <= box[0] or box[3] <= box[1]:
            dropped.append(
                {
                    "object_id": obj.get("id"),
                    "category": category,
                    "source_box": bbox,
                    "reason": "no_visible_area_after_clamp_float32",
                }
            )
            continue
        attributes = obj.get("attributes", {})
        flags = [attributes.get(k, False) for k in ["crowd", "ignored"]]
        if any(not isinstance(flag, bool) for flag in flags):
            raise ValueError("crowd/ignored must be boolean attributes")
        boxes.append(box)
        labels.append(CATEGORY[category])
        crowd.append(int(any(flags)))
    return {
        "boxes": np.asarray(boxes, dtype=np.float32).reshape(-1, 4),
        "labels": np.asarray(labels, dtype=np.int64),
        "iscrowd": np.asarray(crowd, dtype=np.int64),
        "dropped": dropped,
    }


def records_for(split):
    folder = CACHE / split
    manifest = folder / "samples.csv"
    with manifest.open("rb") as stream:
        assert (
            hashlib.file_digest(stream, "sha256").hexdigest()
            == VERIFICATION["splits"][split]["manifest_sha256"]
        )
    with manifest.open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    assert len(rows) == VERIFICATION["splits"][split]["images"]
    records = []
    for index, row in enumerate(rows):
        image, annotation = folder / row["image"], folder / row["annotation"]
        if not image.is_file() or not annotation.is_file():
            raise FileNotFoundError(f"Missing pair: {split}/{row['name']}")
        records.append(
            {
                **row,
                "key": f"{split}/{row['name']}",
                "split": split,
                "index": index,
                "image": image,
                "annotation": annotation,
                "width": int(row["width"]),
                "height": int(row["height"]),
            }
        )
    assert len({r["name"] for r in records}) == len(records)
    return records


RAW_RECORDS = {split: records_for(split) for split in ["train", "val", "test"]}
clean, seen, excluded = {}, {}, []
for split in ["test", "val", "train"]:
    clean[split] = []
    for row in sorted(RAW_RECORDS[split], key=lambda r: r["name"]):
        sha = row["image_sha256"]
        if sha in seen:
            excluded.append(
                {"removed": row["key"], "kept": seen[sha], "image_sha256": sha}
            )
        else:
            seen[sha] = row["key"]
            clean[split].append(row)
assert [len(clean[s]) for s in ["train", "val", "test"]] == [69998, 9998, 20000]
assert len(excluded) == 4
training_records, dev_records, test_records = [
    clean[s] for s in ["train", "val", "test"]
]


def seeded_subset(records, count, seed):
    indices = np.sort(np.random.default_rng(seed).permutation(len(records))[:count])
    return [records[int(i)] for i in indices]


limit = {"smoke": 8, "pilot": 2000, "full": len(training_records)}[MODE]
dev_limit = {"smoke": 8, "pilot": 500, "full": len(dev_records)}[MODE]
selected_train = seeded_subset(training_records, limit, SEED)
selected_dev = seeded_subset(dev_records, dev_limit, SEED + 1)
split_description = {
    "dataset": DATASET_ID,
    "mode": MODE,
    "seed": SEED,
    "train": [r["key"] for r in training_records],
    "dev": [r["key"] for r in dev_records],
    "test": [r["key"] for r in test_records],
    "selected_train": [r["key"] for r in selected_train],
    "selected_dev": [r["key"] for r in selected_dev],
    "excluded": excluded,
    "dedup_priority": ["test", "val", "train"],
    "annotation_protocol": ANNOTATION_PROTOCOL,
    "manifest_hashes": {
        s: VERIFICATION["splits"][s]["manifest_sha256"] for s in RAW_RECORDS
    },
    "test_provenance": "user-supplied legacy test JSON; not verified Detection 2020",
}
SPLIT_HASH = digest(split_description)
record_result(RUN_KEY / "split.json", {**split_description, "hash": SPLIT_HASH})


def packed_annotations(split):
    """Small, reproducible NPZ cache; no image/polygon copies, no pickle."""
    folder = CACHE / "detection_cache"
    folder.mkdir(exist_ok=True)
    path = folder / f"{split}.npz"
    metadata_path = folder / f"{split}.json"
    signature = digest(
        {
            **ANNOTATION_PROTOCOL,
            "manifest": VERIFICATION["splits"][split]["manifest_sha256"],
        }
    )

    # Validate raw labels even when NPZ exists, so stale source changes cannot be hidden.
    def verify_source(row):
        with row["annotation"].open("rb") as stream:
            checksum = hashlib.file_digest(stream, "sha256").hexdigest()
        if checksum != row["annotation_sha256"]:
            raise ValueError(f"BDD annotation SHA256 mismatch: {row['key']}")

    with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 1)) as pool:
        for _ in pool.map(verify_source, RAW_RECORDS[split], buffersize=64):
            pass
    if path.exists() and metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        with path.open("rb") as stream:
            valid = (
                metadata.get("signature") == signature
                and hashlib.file_digest(stream, "sha256").hexdigest()
                == metadata["sha256"]
            )
        if not valid:
            raise ValueError(
                f"Detection cache mismatch: {path}; remove this generated cache to rebuild from source."
            )
    else:
        start = time.perf_counter()
        arrays = {k: [] for k in ["boxes", "labels", "iscrowd"]}
        offsets, dropped = [0], []

        def parse_record(row):
            data = row["annotation"].read_bytes()
            if hashlib.sha256(data).hexdigest() != row["annotation_sha256"]:
                raise ValueError(f"Annotation SHA256 mismatch: {row['key']}")
            target = parse_bdd_annotation(
                json.loads(data), row["width"], row["height"], row["name"]
            )
            assert len(target["labels"]) + len(target["dropped"]) == int(row["boxes"])
            target["key"] = row["key"]
            return target

        print(
            f"Build {split} detection cache ({len(RAW_RECORDS[split]):,} JSON)...",
            flush=True,
        )
        with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 1)) as pool:
            for index, target in enumerate(
                pool.map(parse_record, RAW_RECORDS[split], buffersize=64)
            ):
                for key, values in arrays.items():
                    values.append(target[key])
                offsets.append(offsets[-1] + len(target["labels"]))
                dropped.extend(
                    {"key": target["key"], **entry} for entry in target["dropped"]
                )
                if (index + 1) % 20000 == 0:
                    print(split, index + 1, "parsed", flush=True)
        packed = {k: np.concatenate(values) for k, values in arrays.items()}
        packed["offsets"] = np.asarray(offsets, dtype=np.int64)
        temp = path.with_suffix(".tmp")
        with temp.open("wb") as stream:
            np.savez_compressed(stream, **packed)
        temp.replace(path)
        with path.open("rb") as stream:
            sha = hashlib.file_digest(stream, "sha256").hexdigest()
        write_cache_metadata(
            metadata_path,
            {
                "signature": signature,
                "sha256": sha,
                "images": len(RAW_RECORDS[split]),
                "boxes": int(offsets[-1]),
                "source_boxes": VERIFICATION["splits"][split]["boxes"],
                "dropped_boxes": dropped,
                "crowd_or_ignored": int(packed["iscrowd"].sum()),
                "seconds": time.perf_counter() - start,
                "annotation_protocol": ANNOTATION_PROTOCOL,
            },
        )
        print(
            split, "cache ready", round(time.perf_counter() - start, 1), "s", flush=True
        )
    with np.load(path, allow_pickle=False) as packed:
        result = {key: packed[key] for key in packed.files}
    assert len(result["offsets"]) == len(RAW_RECORDS[split]) + 1
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    assert (
        len(result["boxes"]) + len(metadata["dropped_boxes"])
        == VERIFICATION["splits"][split]["boxes"]
    )
    assert np.isfinite(result["boxes"]).all()
    assert (result["boxes"][:, 2:] > result["boxes"][:, :2]).all()
    return result


PACKED = {split: packed_annotations(split) for split in RAW_RECORDS}


class DetectionDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records, self.augment = records, augment
        self.targets = []
        for image_id, row in enumerate(records, 1):
            arrays = PACKED[row["split"]]
            start, end = arrays["offsets"][row["index"] : row["index"] + 2]
            target = {
                k: torch.from_numpy(arrays[k][start:end])
                for k in ["boxes", "labels", "iscrowd"]
            }
            target["area"] = (target["boxes"][:, 2:] - target["boxes"][:, :2]).prod(1)
            target["image_id"] = torch.tensor(image_id)
            self.targets.append(target)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        data = row["image"].read_bytes()
        if hashlib.sha256(data).hexdigest() != row["image_sha256"]:
            raise ValueError(f"Image SHA256 mismatch: {row['key']}")
        with Image.open(BytesIO(data)) as image:
            assert image.size == (row["width"], row["height"])
            tensor = transforms.functional.to_tensor(image.convert("RGB"))
        target = {k: v.clone() for k, v in self.targets[index].items()}
        if self.augment and random.random() < 0.5:
            tensor = tensor.flip(-1)
            target["boxes"][:, [0, 2]] = tensor.shape[-1] - target["boxes"][:, [2, 0]]
        return tensor, target


def collate_detection(batch):
    return tuple(zip(*batch))


train_set = DetectionDataset(selected_train, augment=True)
dev_set = DetectionDataset(selected_dev)
benchmark_set = DetectionDataset(test_records)
print(
    "BDD100K clean train/dev/test:",
    len(training_records),
    len(dev_records),
    len(test_records),
)
print(
    "Selected train/dev:",
    len(train_set),
    len(dev_set),
    "test evaluation enabled:",
    RUN_TEST,
)
print("Split hash:", SPLIT_HASH, "Excluded duplicate records:", excluded)

## 4. EDA: bbox, lớp và kích thước đối tượng

In [ ]:
counts, areas = np.zeros(NUM_CLASSES, dtype=int), []
for target in train_set.targets:
    counts += np.bincount(target["labels"].numpy(), minlength=NUM_CLASSES)
    areas.extend(target["area"].sqrt().tolist())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(CLASS_NAMES[1:], counts[1:])
axes[0].tick_params(axis="x", rotation=75)
axes[0].set_ylabel("Objects (train subset)")
axes[1].hist(areas, bins=40)
axes[1].set_xlabel("Square root of bounding-box area (pixels)")
show_figure(fig, "eda")

## 5. Paper và giả thuyết
Paper chính: [Lin et al., FPN, CVPR2017](https://openaccess.thecvf.com/content_cvpr_2017/papers/Lin_Feature_Pyramid_Networks_CVPR_2017_paper.pdf).
Top-down truyền semantic từ tầng sâu; lateral nối cùng resolution; 3×3 giảm aliasing.
RPN P2-P6; RoI pooling P2-P5 dựa diện tích. C5 coarse single-scale dùng cùng RoI head
để kiểm soát kiến trúc head. C5 là baseline được chọn, không tự nhận tái hiện C4
trong paper. BDD100K, RoIAlign, resolution, lịch train và một GPU khác cấu hình COCO gốc.

D1: C5; D2: FPN; D3: FPN anchor nhỏ; D4: FPN unfreeze toàn bộ convolutional backbone.
D1-D3 chỉ train C4/C5 (layer3/layer4), FrozenBatchNorm cố định cho tất cả.

Trong Table 3 của paper, ResNet-50 trên COCO minival có AP/AP50: C4 31.6/53.1, C5 28.0/51.7, FPN 33.9/56.9. Đây là kết quả tác giả, không phải số đo local. Paper còn ablate top-down và lateral; notebook này không tái tạo các ablation đó. [Bản arXiv](https://arxiv.org/pdf/1612.03144).

D1/D2 khác cả feature pyramid và anchor coverage (256 so với 32–512). D2-D1 so sánh hai hệ thống; không quy toàn bộ chênh lệch AP cho riêng top-down/lateral. D3-D2 và D4-D2 kiểm tra anchor/unfreeze trong family FPN.


## 6. Backbone ResNet-50 và feature pyramid

In [ ]:
class Bottleneck(nn.Module):
    def __init__(self, incoming, channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(incoming, channels, 1, bias=False)
        self.bn1 = FrozenBatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, stride, 1, bias=False)
        self.bn2 = FrozenBatchNorm2d(channels)
        self.conv3 = nn.Conv2d(channels, channels * 4, 1, bias=False)
        self.bn3 = FrozenBatchNorm2d(channels * 4)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = (
            nn.Sequential(
                nn.Conv2d(incoming, channels * 4, 1, stride, bias=False),
                FrozenBatchNorm2d(channels * 4),
            )
            if stride != 1 or incoming != channels * 4
            else None
        )

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.relu(self.bn2(self.conv2(y)))
        return self.relu(self.bn3(self.conv3(y)) + identity)


class ResNet50Features(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = FrozenBatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        incoming = 64
        for level, (channels, count) in enumerate(
            zip([64, 128, 256, 512], [3, 4, 6, 3]), 1
        ):
            blocks = [Bottleneck(incoming, channels, 1 if level == 1 else 2)]
            blocks += [Bottleneck(channels * 4, channels) for _ in range(count - 1)]
            setattr(self, f"layer{level}", nn.Sequential(*blocks))
            incoming = channels * 4

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        features = []
        for layer in [self.layer1, self.layer2, self.layer3, self.layer4]:
            x = layer(x)
            features.append(x)
        return features

    def load_imagenet(self):
        torch.hub.set_dir(str(ROOT / "datasets" / "pretrained"))
        weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V1.get_state_dict(
            progress=True, check_hash=True
        )
        weights = {
            k: v
            for k, v in weights.items()
            if not k.startswith("fc.") and not k.endswith("num_batches_tracked")
        }
        self.load_state_dict(weights, strict=True)


class FeaturePyramid(nn.Module):
    def __init__(self):
        super().__init__()
        self.lateral = nn.ModuleList(
            [nn.Conv2d(c, 256, 1) for c in [256, 512, 1024, 2048]]
        )
        self.output = nn.ModuleList(
            [nn.Conv2d(256, 256, 3, padding=1) for _ in range(4)]
        )

    def forward(self, features):
        pyramids, top = [], None
        for feature, lateral, output in reversed(
            list(zip(features, self.lateral, self.output))
        ):
            x = lateral(feature)
            if top is not None:
                x = x + F.interpolate(top, size=x.shape[-2:], mode="nearest")
            top = x
            pyramids.insert(0, output(x))
        return pyramids + [F.max_pool2d(pyramids[-1], 1, 2)]

## 7. Anchor, box coding, matching và sampling

In [ ]:
def encode_boxes(gt, anchors, weights=(1.0, 1.0, 1.0, 1.0)):
    anchor_size = (anchors[:, 2:] - anchors[:, :2]).clamp_min(1e-6)
    anchor_center = anchors[:, :2] + anchor_size / 2
    gt_size = (gt[:, 2:] - gt[:, :2]).clamp_min(1e-6)
    gt_center = gt[:, :2] + gt_size / 2
    delta = torch.cat(
        [(gt_center - anchor_center) / anchor_size, torch.log(gt_size / anchor_size)], 1
    )
    return delta * delta.new_tensor(weights)


def decode_boxes(delta, anchors, weights=(1.0, 1.0, 1.0, 1.0)):
    size = (anchors[:, 2:] - anchors[:, :2]).clamp_min(1e-6)
    center = anchors[:, :2] + size / 2
    delta = delta / delta.new_tensor(weights)
    predicted_center = delta[:, :2] * size + center
    predicted_size = delta[:, 2:].clamp(max=math.log(1000.0 / 16)).exp() * size
    return torch.cat(
        [predicted_center - predicted_size / 2, predicted_center + predicted_size / 2],
        1,
    )


def anchors_for_feature(feature, size, stride):
    ratios = feature.new_tensor([0.5, 1.0, 2.0])
    widths, heights = size / ratios.sqrt(), size * ratios.sqrt()
    base = torch.stack([-widths / 2, -heights / 2, widths / 2, heights / 2], 1)
    ys, xs = torch.meshgrid(
        (torch.arange(feature.shape[-2], device=feature.device) + 0.5) * stride,
        (torch.arange(feature.shape[-1], device=feature.device) + 0.5) * stride,
        indexing="ij",
    )
    centers = torch.stack([xs, ys, xs, ys], -1).reshape(-1, 1, 4)
    return (centers + base).reshape(-1, 4)


def intersection_over_anchor(boxes, regions):
    left_top = torch.maximum(boxes[:, None, :2], regions[None, :, :2])
    right_bottom = torch.minimum(boxes[:, None, 2:], regions[None, :, 2:])
    intersection = (right_bottom - left_top).clamp_min(0).prod(2)
    return intersection / box_area(boxes).clamp_min(1e-6)[:, None]


def assign_boxes(boxes, target, low, high, low_quality=False):
    normal = target["iscrowd"] == 0
    gt, gt_labels = target["boxes"][normal], target["labels"][normal]
    labels = torch.zeros(len(boxes), device=boxes.device, dtype=torch.long)
    matched = torch.zeros(len(boxes), device=boxes.device, dtype=torch.long)
    if len(gt):
        iou = box_iou(boxes, gt)
        best, matched = iou.max(1)
        labels[(best >= low) & (best < high)] = -1
        positive = best >= high
        if low_quality:
            positive |= (iou == iou.max(0).values[None]).any(1)
        labels[positive] = gt_labels[matched[positive]]
    ignored = target["boxes"][~normal]
    if len(ignored):
        overlap = intersection_over_anchor(boxes, ignored).max(1).values >= 0.5
        labels[overlap & (labels <= 0)] = -1
    return labels, matched, gt


def sample_labels(labels, count, positive_fraction):
    pos, neg = torch.where(labels > 0)[0], torch.where(labels == 0)[0]
    npos = min(len(pos), int(count * positive_fraction))
    nneg = min(len(neg), count - npos)
    pos = pos[torch.randperm(len(pos), device=labels.device)[:npos]]
    neg = neg[torch.randperm(len(neg), device=labels.device)[:nneg]]
    return torch.cat([pos, neg]), pos

## 8. Region proposal network tự định nghĩa

In [ ]:
class RegionProposalNetwork(nn.Module):
    def __init__(self, sizes, levels):
        super().__init__()
        self.sizes, self.levels = sizes, levels
        self.conv = nn.Conv2d(256, 256, 3, padding=1)
        self.objectness = nn.Conv2d(256, 3, 1)
        self.regression = nn.Conv2d(256, 12, 1)
        for layer in [self.conv, self.objectness, self.regression]:
            nn.init.normal_(layer.weight, std=0.01)
            nn.init.zeros_(layer.bias)

    def forward(self, features, image_sizes, targets=None):
        logits, deltas, anchor_levels = [], [], []
        for feature, size, level in zip(features, self.sizes, self.levels):
            hidden = F.relu(self.conv(feature))
            logits.append(
                self.objectness(hidden)
                .permute(0, 2, 3, 1)
                .reshape(feature.shape[0], -1)
            )
            deltas.append(
                self.regression(hidden)
                .reshape(feature.shape[0], 3, 4, *feature.shape[-2:])
                .permute(0, 3, 4, 1, 2)
                .reshape(feature.shape[0], -1, 4)
            )
            anchor_levels.append(anchors_for_feature(feature, size, 2**level))
        all_logits, all_deltas = torch.cat(logits, 1), torch.cat(deltas, 1)
        all_anchors = torch.cat(anchor_levels)
        proposals = []
        for batch_index, image_size in enumerate(image_sizes):
            boxes, scores, level_ids = [], [], []
            for level_index, (score, delta, anchors) in enumerate(
                zip(logits, deltas, anchor_levels)
            ):
                k = min(2000 if self.training else 1000, anchors.shape[0])
                idx = score[batch_index].detach().topk(k).indices
                box = decode_boxes(
                    delta[batch_index, idx].detach().float(), anchors[idx].float()
                )
                box = clip_boxes_to_image(box, image_size)
                valid = torch.isfinite(box).all(1) & (
                    (box[:, 2:] - box[:, :2]) >= 1
                ).all(1)
                boxes.append(box[valid])
                scores.append(score[batch_index, idx][valid].detach().float().sigmoid())
                level_ids.append(
                    torch.full((int(valid.sum()),), level_index, device=box.device)
                )
            boxes, scores, level_ids = (
                torch.cat(boxes),
                torch.cat(scores),
                torch.cat(level_ids),
            )
            keep = batched_nms(boxes, scores, level_ids, 0.7)[
                : 2000 if self.training else 1000
            ]
            proposals.append(boxes[keep])
        loss_objectness = all_logits.sum() * 0
        loss_regression = all_deltas.sum() * 0
        if self.training:
            for i, target in enumerate(targets):
                labels, matched, gt = assign_boxes(
                    all_anchors, target, 0.3, 0.7, low_quality=True
                )
                indices, positive = sample_labels(labels, 256, 0.5)
                denominator = max(1, len(indices))
                loss_objectness += (
                    F.binary_cross_entropy_with_logits(
                        all_logits[i, indices],
                        (labels[indices] > 0).float(),
                        reduction="sum",
                    )
                    / denominator
                )
                if len(positive):
                    regression_target = encode_boxes(
                        gt[matched[positive]], all_anchors[positive]
                    )
                    loss_regression += (
                        F.smooth_l1_loss(
                            all_deltas[i, positive],
                            regression_target,
                            beta=1 / 9,
                            reduction="sum",
                        )
                        / denominator
                    )
            loss_objectness /= len(targets)
            loss_regression /= len(targets)
        return proposals, {
            "rpn_objectness": loss_objectness,
            "rpn_box": loss_regression,
        }

## 9. RoIAlign và head chung

In [ ]:
class RoIHead(nn.Module):
    def __init__(self, levels):
        super().__init__()
        self.levels = levels
        self.fc1 = nn.Linear(256 * 7 * 7, 1024)
        self.fc2 = nn.Linear(1024, 1024)
        self.classification = nn.Linear(1024, NUM_CLASSES)
        self.regression = nn.Linear(1024, NUM_CLASSES * 4)
        nn.init.normal_(self.classification.weight, std=0.01)
        nn.init.normal_(self.regression.weight, std=0.001)
        nn.init.zeros_(self.classification.bias)
        nn.init.zeros_(self.regression.bias)

    def pool(self, features, proposals):
        rois = torch.cat(
            [
                torch.cat([p.new_full((len(p), 1), i), p], 1)
                for i, p in enumerate(proposals)
            ]
        )
        if len(rois) == 0:
            return features[0].new_empty((0, 256, 7, 7))
        if len(self.levels) == 1:
            assignment = torch.full((len(rois),), self.levels[0], device=rois.device)
        else:
            size = (
                ((rois[:, 3] - rois[:, 1]) * (rois[:, 4] - rois[:, 2]))
                .clamp_min(1e-6)
                .sqrt()
            )
            assignment = torch.floor(4 + torch.log2(size / 224 + 1e-6)).clamp(2, 5)
        pooled = features[0].new_zeros((len(rois), 256, 7, 7))
        for feature, level in zip(features, self.levels):
            indices = torch.where(assignment == level)[0]
            if len(indices):
                pooled[indices] = roi_align(
                    feature,
                    rois[indices],
                    output_size=7,
                    spatial_scale=1 / (2**level),
                    sampling_ratio=2,
                    aligned=True,
                )
        return pooled

    def forward(self, features, proposals, image_sizes, targets=None):
        labels, regression_targets = [], []
        if self.training:
            sampled = []
            for proposal, target in zip(proposals, targets):
                gt = target["boxes"][target["iscrowd"] == 0]
                proposal = torch.cat([proposal, gt])
                label, matched, gt = assign_boxes(proposal, target, 0.5, 0.5)
                indices, positive = sample_labels(label, 512, 0.25)
                regression = proposal.new_zeros((len(proposal), 4))
                if len(positive):
                    regression[positive] = encode_boxes(
                        gt[matched[positive]],
                        proposal[positive],
                        (10.0, 10.0, 5.0, 5.0),
                    )
                sampled.append(proposal[indices])
                labels.append(label[indices])
                regression_targets.append(regression[indices])
            proposals = sampled
        pooled = self.pool(features, proposals).flatten(1)
        hidden = F.relu(self.fc2(F.relu(self.fc1(pooled))))
        logits, deltas = (
            self.classification(hidden),
            self.regression(hidden).reshape(-1, NUM_CLASSES, 4),
        )
        if self.training:
            labels, regression_targets = (
                torch.cat(labels),
                torch.cat(regression_targets),
            )
            positive = torch.where(labels > 0)[0]
            cls_loss = F.cross_entropy(logits, labels, reduction="sum") / max(
                1, len(labels)
            )
            reg_loss = F.smooth_l1_loss(
                deltas[positive, labels[positive]],
                regression_targets[positive],
                beta=1 / 9,
                reduction="sum",
            ) / max(1, len(labels))
            return None, {"roi_classification": cls_loss, "roi_box": reg_loss}
        predictions = []
        for boxes, logit, delta, image_size in zip(
            proposals,
            logits.split([len(p) for p in proposals]),
            deltas.split([len(p) for p in proposals]),
            image_sizes,
        ):
            scores = logit.float().softmax(1)[:, 1:].reshape(-1)
            labels = (
                torch.arange(1, NUM_CLASSES, device=boxes.device)
                .expand(len(boxes), -1)
                .reshape(-1)
            )
            decoded = decode_boxes(
                delta[:, 1:].reshape(-1, 4).float(),
                boxes[:, None].expand(-1, NUM_CLASSES - 1, -1).reshape(-1, 4),
                (10.0, 10.0, 5.0, 5.0),
            )
            decoded = clip_boxes_to_image(decoded, image_size)
            valid = (
                (scores >= 0.001)
                & torch.isfinite(decoded).all(1)
                & ((decoded[:, 2:] - decoded[:, :2]) >= 1).all(1)
            )
            decoded, scores, labels = decoded[valid], scores[valid], labels[valid]
            keep = batched_nms(decoded, scores, labels, 0.5)[:100]
            predictions.append(
                {"boxes": decoded[keep], "scores": scores[keep], "labels": labels[keep]}
            )
        return predictions, {}

## 10. Transform và detector end-to-end

In [ ]:
class FasterRCNN(nn.Module):
    def __init__(self, variant, pretrained=True, short_side=640, max_side=1280):
        super().__init__()
        self.variant, self.short_side, self.max_side = variant, short_side, max_side
        self.backbone = ResNet50Features()
        if pretrained:
            self.backbone.load_imagenet()
        if variant == "D1":
            self.neck = nn.Conv2d(2048, 256, 1)
            self.rpn = RegionProposalNetwork([256], [5])
            self.roi = RoIHead([5])
        else:
            self.neck = FeaturePyramid()
            self.rpn = RegionProposalNetwork(
                [16, 32, 64, 128, 256] if variant == "D3" else [32, 64, 128, 256, 512],
                [2, 3, 4, 5, 6],
            )
            self.roi = RoIHead([2, 3, 4, 5])
        for name, p in self.backbone.named_parameters():
            p.requires_grad_(variant == "D4" or name.startswith(("layer3.", "layer4.")))

    def prepare(self, images, targets):
        resized, sizes, originals, updated = [], [], [], []
        for i, image in enumerate(images):
            height, width = image.shape[-2:]
            scale = min(
                self.short_side / min(height, width), self.max_side / max(height, width)
            )
            image = F.interpolate(
                image[None],
                scale_factor=scale,
                mode="bilinear",
                align_corners=False,
                recompute_scale_factor=True,
            )[0]
            new_h, new_w = image.shape[-2:]
            originals.append((height, width))
            sizes.append((new_h, new_w))
            image = (
                image - image.new_tensor([0.485, 0.456, 0.406])[:, None, None]
            ) / image.new_tensor([0.229, 0.224, 0.225])[:, None, None]
            resized.append(image)
            if targets is not None:
                target = {k: v.clone() for k, v in targets[i].items()}
                target["boxes"] = target["boxes"] * image.new_tensor(
                    [new_w / width, new_h / height, new_w / width, new_h / height]
                )
                updated.append(target)
        pad_h = math.ceil(max(s[0] for s in sizes) / 32) * 32
        pad_w = math.ceil(max(s[1] for s in sizes) / 32) * 32
        batch = resized[0].new_zeros((len(images), 3, pad_h, pad_w))
        for i, image in enumerate(resized):
            batch[i, :, : image.shape[1], : image.shape[2]] = image
        return batch, sizes, originals, updated if targets is not None else None

    def forward(self, images, targets=None):
        if self.training and targets is None:
            raise ValueError("Training needs targets")
        batch, sizes, originals, targets = self.prepare(images, targets)
        backbone = self.backbone(batch)
        features = (
            [self.neck(backbone[-1])] if self.variant == "D1" else self.neck(backbone)
        )
        proposals, rpn_losses = self.rpn(features, sizes, targets)
        predictions, roi_losses = self.roi(
            features[: len(self.roi.levels)], proposals, sizes, targets
        )
        if self.training:
            return {**rpn_losses, **roi_losses}
        for prediction, resized, original in zip(predictions, sizes, originals):
            rh, rw = resized
            oh, ow = original
            prediction["boxes"] *= prediction["boxes"].new_tensor(
                [ow / rw, oh / rh, ow / rw, oh / rh]
            )
        return predictions

## 11. COCO-style AP50:95, per-class AP và PR
IoU 0.50–0.95 bước 0.05, 101 recall points, maxDets 1/10/100.
Best theo dev AP50:95. AP50/AP75/size/AR100 bổ sung; lớp không có GT hợp lệ
trong subset có AP `null`, không gán bằng 0 và không đưa vào mean.
PR curve là precision nội suy ở IoU0.50, area=all, maxDets=100.


In [ ]:
def coco_metrics(targets, predictions, image_sizes):
    coco = COCO()
    annotations, images, detections = [], [], []
    annotation_id = 1
    for target, prediction, (height, width) in zip(targets, predictions, image_sizes):
        image_id = int(target["image_id"])
        images.append({"id": image_id, "height": height, "width": width})
        for box, label, crowd in zip(
            target["boxes"], target["labels"], target["iscrowd"]
        ):
            x1, y1, x2, y2 = box.tolist()
            annotations.append(
                {
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": int(label),
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "area": (x2 - x1) * (y2 - y1),
                    "iscrowd": int(crowd),
                }
            )
            annotation_id += 1
        for box, label, score in zip(
            prediction["boxes"], prediction["labels"], prediction["scores"]
        ):
            x1, y1, x2, y2 = box.tolist()
            detections.append(
                {
                    "image_id": image_id,
                    "category_id": int(label),
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": float(score),
                }
            )
    coco.dataset = {
        "info": {},
        "images": images,
        "annotations": annotations,
        "categories": [
            {"id": i, "name": name} for i, name in enumerate(CLASS_NAMES) if i
        ],
    }
    coco.createIndex()
    if detections:
        result = coco.loadRes(detections)
    else:
        result = COCO()
        result.dataset = {**coco.dataset, "annotations": []}
        result.createIndex()
    evaluator = COCOeval(coco, result, "bbox")
    evaluator.params.iouThrs = np.linspace(0.5, 0.95, 10)
    evaluator.params.recThrs = np.linspace(0, 1, 101)
    evaluator.params.maxDets = [1, 10, 100]
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()
    metrics = {
        "ap": float(evaluator.stats[0]),
        "ap50": float(evaluator.stats[1]),
        "ap75": float(evaluator.stats[2]),
        "ap_small": float(evaluator.stats[3]),
        "ap_medium": float(evaluator.stats[4]),
        "ap_large": float(evaluator.stats[5]),
        "ar100": float(evaluator.stats[8]),
    }
    per_class, curves = {}, {}
    precision = evaluator.eval["precision"][:, :, :, 0, -1]
    for index, category in enumerate(evaluator.params.catIds):
        values = precision[:, :, index]
        valid = values[values >= 0]
        row50, row75 = values[0], values[5]
        per_class[CLASS_NAMES[category]] = {
            "ap": float(valid.mean()) if len(valid) else None,
            "ap50": float(row50[row50 >= 0].mean()) if (row50 >= 0).any() else None,
            "ap75": float(row75[row75 >= 0].mean()) if (row75 >= 0).any() else None,
            "gt_instances": sum(
                int(((t["labels"] == category) & (t["iscrowd"] == 0)).sum())
                for t in targets
            ),
        }
        if (row50 >= 0).any():
            curves[CLASS_NAMES[category]] = {
                "recall": evaluator.params.recThrs[row50 >= 0].tolist(),
                "precision": row50[row50 >= 0].tolist(),
            }
    return metrics, per_class, curves


@torch.no_grad()
def evaluate_detector(model, dataset, batch_size=2):
    model.eval()
    targets_all, predictions_all, sizes = [], [], []
    elapsed, count = 0.0, 0
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
        collate_fn=collate_detection,
    )
    for images, targets in loader:
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        output = model([image.to(DEVICE) for image in images])
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elapsed += time.perf_counter() - start
        count += len(images)
        predictions_all.extend(
            [{k: v.cpu() for k, v in prediction.items()} for prediction in output]
        )
        targets_all.extend(targets)
        sizes.extend([tuple(image.shape[-2:]) for image in images])
    scores, per_class, curves = coco_metrics(targets_all, predictions_all, sizes)
    metrics = {
        "fps_batch": count / max(elapsed, 1e-9),
        "inference_seconds": elapsed,
        "images": count,
        "inference_batch": batch_size,
        **scores,
    }
    return metrics, per_class, curves, predictions_all

## 12. Các kiểm tra thuật toán và forward/backward

In [ ]:
anchors = torch.tensor([[0.0, 0.0, 20.0, 30.0], [10.0, 20.0, 50.0, 60.0]])
gt = torch.tensor([[2.0, 3.0, 23.0, 31.0], [11.0, 18.0, 48.0, 62.0]])
torch.testing.assert_close(
    decode_boxes(encode_boxes(gt, anchors), anchors), gt, atol=1e-5, rtol=1e-5
)
empty = {
    "boxes": torch.empty(0, 4),
    "labels": torch.empty(0, dtype=torch.long),
    "iscrowd": torch.empty(0, dtype=torch.long),
}
labels, _, _ = assign_boxes(anchors, empty, 0.3, 0.7)
assert (labels == 0).all()
crowd = {
    "boxes": anchors[:1],
    "labels": torch.tensor([3]),
    "iscrowd": torch.tensor([1]),
}
labels, _, _ = assign_boxes(anchors, crowd, 0.3, 0.7)
assert labels[0] == -1
features = [
    torch.randn(1, c, h, w)
    for c, h, w in [(256, 25, 37), (512, 13, 19), (1024, 7, 10), (2048, 4, 5)]
]
pyramids = FeaturePyramid()(features)
assert [p.shape[-2:] for p in pyramids] == [(25, 37), (13, 19), (7, 10), (4, 5), (2, 3)]
t = {
    "boxes": gt[:1],
    "labels": torch.tensor([3]),
    "iscrowd": torch.tensor([0]),
    "image_id": torch.tensor(1),
}
perfect = {
    "boxes": gt[:1].clone(),
    "labels": torch.tensor([3]),
    "scores": torch.tensor([0.9]),
}
metrics, per_class, _ = coco_metrics([t], [perfect], [(100, 100)])
assert abs(metrics["ap"] - 1) < 1e-6
assert per_class["car"]["ap"] > 0.999
assert per_class["train"]["ap"] is None
bad = {
    "boxes": torch.cat([torch.tensor([[80.0, 80.0, 90.0, 90.0]]), gt[:1]]),
    "labels": torch.tensor([3, 3]),
    "scores": torch.tensor([0.95, 0.9]),
}
assert coco_metrics([t], [bad], [(100, 100)])[0]["ap"] < 1
no_detections = {
    "boxes": torch.empty(0, 4),
    "labels": torch.empty(0, dtype=torch.long),
    "scores": torch.empty(0),
}
assert coco_metrics([t], [no_detections], [(100, 100)])[0]["ap"] == 0
fixture = {
    "name": "fixture",
    "frames": [
        {
            "objects": [
                {
                    "category": "person",
                    "box2d": {"x1": 10, "y1": 20, "x2": 10, "y2": 20},
                    "attributes": {"occluded": True, "truncated": True},
                },
                {
                    "category": "bike",
                    "box2d": {"x1": 90, "y1": 90, "x2": 100, "y2": 100},
                },
                {
                    "category": "motor",
                    "box2d": {"x1": 1, "y1": 2, "x2": 3, "y2": 4},
                    "attributes": {"crowd": True},
                },
                {"category": "area/drivable", "poly2d": []},
            ]
        }
    ],
}
parsed = parse_bdd_annotation(fixture, 100, 100, "fixture")
np.testing.assert_array_equal(parsed["boxes"][0], [10, 20, 11, 21])
np.testing.assert_array_equal(parsed["boxes"][1], [90, 90, 100, 100])
assert parsed["labels"].tolist() == [
    CATEGORY["pedestrian"],
    CATEGORY["bicycle"],
    CATEGORY["motorcycle"],
]
assert parsed["iscrowd"].tolist() == [0, 0, 1]
fixture["frames"][0]["objects"].append(
    {"category": "car", "box2d": {"x1": 100, "y1": 10, "x2": 100, "y2": 20}}
)
boundary = parse_bdd_annotation(fixture, 100, 100)
assert len(boundary["labels"]) == 3 and len(boundary["dropped"]) == 1
assert NUM_CLASSES == 11
# Same-class prediction on an explicit crowd region must not become an FP.
crowd_truth = {
    **t,
    "boxes": torch.cat([t["boxes"], torch.tensor([[80.0, 80.0, 90.0, 90.0]])]),
    "labels": torch.tensor([3, 3]),
    "iscrowd": torch.tensor([0, 1]),
}
assert coco_metrics([crowd_truth], [bad], [(100, 100)])[0]["ap"] > 0.999
print("Box coding, empty GT, crowd, odd FPN shapes and AP checks passed")

# Actual image forward/backward for every variant; at 128 px for technical tests only.
for variant in ["D1", "D2", "D3", "D4"]:
    model = FasterRCNN(variant, pretrained=False, short_side=128, max_side=256).to(
        DEVICE
    )
    image, target = dev_set[0]
    model.train()
    losses = model([image.to(DEVICE)], [{k: v.to(DEVICE) for k, v in target.items()}])
    assert all(torch.isfinite(loss) for loss in losses.values())
    sum(losses.values()).backward()
    assert (
        model.neck.weight.grad is not None
        if variant == "D1"
        else model.neck.lateral[0].weight.grad is not None
    )
    model.eval()
    with torch.no_grad():
        prediction = model([image.to(DEVICE)])[0]
    assert prediction["boxes"].shape[1] == 4 and len(prediction["boxes"]) <= 100
    del model, losses, prediction
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
print("All four detector variants passed forward/backward/inference")

# Check AMP/backward at the actual full-run resolution on two real images.
if MODE == "smoke" and DEVICE.type == "cuda":
    capacity = []
    for variant in ["D1", "D2", "D3", "D4"]:
        model = FasterRCNN(variant, pretrained=True).to(DEVICE)
        examples = [dev_set[i] for i in [0, 1]]
        torch.cuda.reset_peak_memory_stats()
        model.train()
        with torch.autocast("cuda"):
            losses = model(
                [image.to(DEVICE) for image, _ in examples],
                [
                    {k: v.to(DEVICE) for k, v in target.items()}
                    for _, target in examples
                ],
            )
        sum(losses.values()).backward()
        assert all(
            torch.isfinite(p.grad).all()
            for p in model.parameters()
            if p.grad is not None
        )
        capacity.append(
            {
                "variant": variant,
                "short_side": 640,
                "micro_batch": 2,
                "peak_vram_mb": torch.cuda.max_memory_allocated() / 2**20,
            }
        )
        del model, losses
        gc.collect()
        torch.cuda.empty_cache()
    record_result(RUN_KEY / "capacity_check.json", capacity)
    print("Full-resolution capacity check:", capacity)

if RUN_OVERFIT_CHECK:
    seed_all(SEED)
    model = FasterRCNN("D2", pretrained=True, short_side=128, max_side=256).to(DEVICE)
    sample_index = next(
        i
        for i, target in enumerate(dev_set.targets)
        if ((target["iscrowd"] == 0) & (target["labels"] > 0)).any()
    )
    image, target = dev_set[sample_index]
    images = [image.to(DEVICE)]
    targets = [{k: v.to(DEVICE) for k, v in target.items()}]
    optimizer_check = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad], lr=0.005, momentum=0.9
    )
    overfit_losses = []
    for step in range(30):
        model.train()
        optimizer_check.zero_grad(set_to_none=True)
        loss = sum(model(images, targets).values())
        assert torch.isfinite(loss)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 10)
        optimizer_check.step()
        overfit_losses.append(float(loss.detach()))
    initial, final = (
        float(np.mean(overfit_losses[:5])),
        float(np.mean(overfit_losses[-5:])),
    )
    assert final < initial, (initial, final)
    record_result(
        RUN_KEY / "overfit_check.json",
        {
            "technical_only": True,
            "n_images": 1,
            "steps": 30,
            "initial_loss_mean5": initial,
            "final_loss_mean5": final,
        },
    )
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print("Tiny-image optimization check:", initial, "->", final)

## 13. Engine SGD, AMP, accumulation, checkpoint
Early stopping theo AP50:95 của validation, patience4 và min_delta=0,0001.
Bộ đếm được reset khi AP tăng hơn min_delta so với mốc theo dõi; dừng sau4 epoch
liên tiếp không đạt mức này. Checkpoint giữ AP validation cao nhất tuyệt đối,
kể cả cải thiện nhỏ hơn min_delta. Test không được dùng để chọn điểm dừng.


In [ ]:
def train_detector(variant):
    seed_all(SEED)
    folder = RUN_KEY / variant

    short_side = 128 if MODE == "smoke" else 640
    max_side = 256 if MODE == "smoke" else 1280
    epochs = {"smoke": 1, "pilot": 2, "full": 12}[MODE]
    protocol = {
        "variant": variant,
        "seed": SEED,
        "mode": MODE,
        "split": SPLIT_HASH,
        "dataset": DATASET_ID,
        "classes": CLASS_NAMES,
        "annotation_protocol": ANNOTATION_PROTOCOL,
        "selection_metric": "dev_ap50_95",
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "early_stop_min_delta": EARLY_STOP_MIN_DELTA,
        "early_stop_mode": "max",
        "eval_iou": [0.5, 0.95, 0.05],
        "eval_recall_points": 101,
        "eval_score_threshold": 0.001,
        "eval_nms": 0.5,
        "max_detections": 100,
        "micro_batch": MICRO_BATCH,
        "accumulation": ACCUM,
        "train_images": len(train_set),
        "dev_images": len(dev_set),
        "epochs": epochs,
        "effective_batch": MICRO_BATCH * ACCUM,
        "resize": [short_side, max_side],
        "optimizer": "SGD",
        "lr_head": 0.005,
        "lr_backbone": 0.0005,
        "momentum": 0.9,
        "weight_decay": 1e-4,
        "steps": [8, 11],
        "warmup_steps": 500,
        "versions": VERSIONS,
    }
    config_hash = digest(protocol)

    record_result(folder / "protocol", protocol)
    model = FasterRCNN(
        variant, pretrained=True, short_side=short_side, max_side=max_side
    ).to(DEVICE)
    groups = [
        {
            "params": [p for p in model.backbone.parameters() if p.requires_grad],
            "lr": 0.0005,
        },
        {
            "params": list(model.neck.parameters())
            + list(model.rpn.parameters())
            + list(model.roi.parameters()),
            "lr": 0.005,
        },
    ]
    optimizer = torch.optim.SGD(groups, momentum=0.9, weight_decay=1e-4)
    scaler = torch.amp.GradScaler(
        "cuda", enabled=DEVICE.type == "cuda", init_scale=1024
    )
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[8, 11], gamma=0.1
    )
    start, history, best, updates, completed = 0, [], -1.0, 0, False
    monitor_ap, bad_epochs = -math.inf, 0
    stopping_reason = "max_epochs"

    loader = DataLoader(
        train_set,
        batch_size=MICRO_BATCH,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=collate_detection,
    )
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(start, epochs if not completed else start):
        model.train()
        epoch_start = time.perf_counter()
        sums, count, window_images = {}, 0, 0
        optimizer.zero_grad(set_to_none=True)
        scheduled = scheduler.get_last_lr()
        for step, (images, targets) in enumerate(loader):
            warmup = min(1.0, (updates + 1) / 500)
            for group, lr in zip(optimizer.param_groups, scheduled):
                group["lr"] = lr * warmup
            images = [image.to(DEVICE) for image in images]
            targets = [
                {k: v.to(DEVICE) for k, v in target.items()} for target in targets
            ]
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                losses = model(images, targets)
                loss = sum(losses.values()) * len(images)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite detection loss: {variant}")
            scaler.scale(loss).backward()
            window_images += len(images)
            for key, value in losses.items():
                sums[key] = sums.get(key, 0.0) + value.item() * len(images)
            count += len(images)
            if (step + 1) % ACCUM == 0 or step + 1 == len(loader):
                scaler.unscale_(optimizer)
                for p in model.parameters():
                    if p.grad is not None:
                        p.grad.div_(window_images)
                nn.utils.clip_grad_norm_(model.parameters(), 10)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                updates += int(scaler.get_scale() >= old_scale)
                optimizer.zero_grad(set_to_none=True)
                window_images = 0
        # Restore epoch scheduled rates before stepping; warmup does not compound the scheduler.
        for group, lr in zip(optimizer.param_groups, scheduled):
            group["lr"] = lr
        scheduler.step()
        metrics, _, _, _ = evaluate_detector(model, dev_set, MICRO_BATCH)
        current_ap = float(metrics["ap"])
        if not math.isfinite(current_ap):
            raise FloatingPointError(f"Non-finite validation AP: {variant}")
        if current_ap > monitor_ap + EARLY_STOP_MIN_DELTA:
            monitor_ap, bad_epochs = current_ap, 0
        else:
            bad_epochs += 1
        row = {
            "epoch": epoch + 1,
            **{k: v / count for k, v in sums.items()},
            "dev_ap": metrics["ap"],
            "dev_ap50": metrics["ap50"],
            "early_stop_counter": bad_epochs,
            "seconds": time.perf_counter() - epoch_start,
        }
        history.append(row)
        if metrics["ap"] > best:
            best = metrics["ap"]
            remember_checkpoint(
                folder / "best (RAM)",
                {
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "config_hash": config_hash,
                },
            )
        completed = bad_epochs >= EARLY_STOP_PATIENCE or epoch + 1 == epochs

        record_result(folder / "history", history)
        print(
            variant,
            row,
            f"early stop={bad_epochs}/{EARLY_STOP_PATIENCE} · min_delta={EARLY_STOP_MIN_DELTA:g}",
            flush=True,
        )
        if bad_epochs >= EARLY_STOP_PATIENCE:
            stopping_reason = "early_stopping"
            print(
                f"{variant}: early stopping tại epoch{epoch + 1}; "
                f"{bad_epochs} epoch không cải thiện validation AP đủ min_delta; "
                "nạp checkpoint có AP validation cao nhất.",
                flush=True,
            )
            break
    best_state = load_checkpoint(folder / "best (RAM)")
    model.load_state_dict(best_state["model"])
    metrics, per_class, curves, predictions = evaluate_detector(
        model, dev_set, MICRO_BATCH
    )
    result = {
        "run": variant,
        "mode": MODE,
        "epochs": len(history),
        "optimizer_updates": updates,
        "stopping_reason": stopping_reason,
        **{
            f"dev_{k}": metrics[k]
            for k in [
                "ap",
                "ap50",
                "ap75",
                "ap_small",
                "ap_medium",
                "ap_large",
                "ar100",
            ]
        },
        "parameters": sum(p.numel() for p in model.parameters()),
        "seconds": sum(r["seconds"] for r in history),
        "fps_batch": metrics["fps_batch"],
        "peak_vram_mb": torch.cuda.max_memory_allocated() / 2**20
        if DEVICE.type == "cuda"
        else 0,
        "best_epoch": best_state["epoch"] + 1,
    }
    if RUN_TEST:
        test, test_class, test_curves, _ = evaluate_detector(
            model, benchmark_set, MICRO_BATCH
        )
        record_result(folder / "test.json", test)
        record_result(folder / "test_per_class.json", test_class)
        record_result(folder / "test_pr_curves.json", test_curves)
        result.update(
            {
                f"test_{k}": test[k]
                for k in [
                    "ap",
                    "ap50",
                    "ap75",
                    "ap_small",
                    "ap_medium",
                    "ap_large",
                    "ar100",
                ]
            }
        )
    record_result(folder / "summary", result)
    record_result(folder / "dev_per_class.json", per_class)
    record_result(folder / "dev_pr_curves.json", curves)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for key in ["rpn_objectness", "rpn_box", "roi_classification", "roi_box"]:
        axes[0].plot(
            [r["epoch"] for r in history], [r[key] for r in history], label=key
        )
    axes[0].legend()
    axes[1].plot(
        [r["epoch"] for r in history],
        [r["dev_ap"] for r in history],
        label="BDD dev AP50:95",
    )
    axes[1].legend()
    show_figure(fig, variant + "_curves")
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for index, ax in enumerate(axes.flat):
        image, target = dev_set[index % len(dev_set)]
        ax.imshow(image.permute(1, 2, 0))
        for box in target["boxes"]:
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(
                Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    edgecolor="#22c55e",
                    linewidth=1.5,
                )
            )
        for box, label, score in zip(
            predictions[index % len(dev_set)]["boxes"],
            predictions[index % len(dev_set)]["labels"],
            predictions[index % len(dev_set)]["scores"],
        ):
            if score < 0.3:
                continue
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(
                Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    edgecolor="#ef4444",
                    linewidth=1,
                )
            )
            ax.text(
                x1,
                y1,
                f"{CLASS_NAMES[int(label)]} {score:.2f}",
                fontsize=6,
                color="white",
                bbox={"facecolor": "#164e9b", "alpha": 0.8, "pad": 1},
            )
        ax.axis("off")
    show_figure(fig, variant + "_predictions_dev")
    fig, ax = plt.subplots(figsize=(7, 4))
    for category in curves:
        ax.plot(
            curves[category]["recall"], curves[category]["precision"], label=category
        )
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(fontsize=8)
    show_figure(fig, variant + "_pr_dev")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result

## 14. Bốn run với cùng split và protocol

In [ ]:
results = []
for variant in ["D1", "D2", "D3", "D4"]:
    try:
        results.append(train_detector(variant))
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError(
            "GPU memory insufficient. Restart kernel, use MICRO_BATCH=1, ACCUM=4; "
            "keep resize and effective batch equal across variants. Resume uses last completed epoch."
        ) from error
record_result(RUN_KEY / "comparison", results)
record_result(RUN_KEY / "comparison", results)
show_table(
    results,
    ["run", "dev_ap", "dev_ap50", "dev_ap75", "parameters", "seconds"],
    "results",
)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([r["run"] for r in results], [r["dev_ap"] for r in results], color="#164e9b")
ax.set_ylabel("BDD100K dev AP50:95 (local legacy labels)")
show_figure(fig, "comparison")
print("[KẾT QUẢ] Bảng/biểu đồ hiển thị trong output cell; không tạo thư mục kết quả.")
if RUN_TEST:
    show_table(results, ["run", "test_ap", "test_ap50", "test_ap75"], "test_results")
finish_notebook()

## 15. Phân tích cần bổ sung từ full run
So sánh hệ thống D2-D1 với giới hạn khác biệt anchor coverage; D3-D2 cho anchor nhỏ; D4-D2 cho unfreeze.
Phân tích per-class AP, PR, object-size AP và ảnh false positive/false negative.
Vẽ GT xanh/prediction đỏ; threshold 0.3 chỉ để hiển thị, eval vẫn threshold 0.001.
FPS bao gồm H2D/resize/model/NMS và đồng bộ CUDA, không gồm đọc disk/COCOeval;
ghi batch và GPU rõ. Chỉ dùng số liệu từ lần chạy đầy đủ để diễn giải kết quả.
Phân tích lớp hiếm (train/rider/motorcycle), small objects, occlusion và cảnh đêm từ nhãn train/dev.
Không so sánh số local legacy với Detection 2020 như cùng benchmark.


## Tái lập và lưu kết quả

Mở notebook trong source_code, chọn kernel **Python 3.14 có PyTorch CUDA** rồi
**Restart Kernel → Run All**. Code dùng toàn bộ dữ liệu local trong datasets;
không đọc code, checkpoint hoặc kết quả của notebook khác.

Mỗi lần Run All khởi tạo model mới. Best checkpoint được sao chép sang **CPU RAM**
để không bị training sửa trọng số, rồi nạp để đánh giá sau khi chọn bằng validation.
Checkpoint không được ghi ra đĩa và không có resume giữa hai phiên kernel.
Bảng, đồ thị, tiến trình và nhận xét hiển thị trong output cell.
**Save notebook sau khi chạy** để giữ kết quả trong .ipynb trước khi push Git.
Restart Kernel sẽ xóa các biến/checkpoint/log trong RAM.

Không tạo thư mục outputs, không tự ghi CSV/PNG/weights/LaTeX vào project.
report/source_latex/assets dành cho báo cáo, được biên tập riêng sau khi có kết quả.
Dataset và pretrained cache chỉ giữ local; xem [datasets/README](../datasets/README.md).
Nguồn đề bài nằm trong [assignment](../assignment/).
